# Portfolio Project: Visualization in Python - Canada Immigration Analytics

---

# 01 — Data Preparation for Canadian Immigration Visual Analytics

## Project purpose

This notebook prepares the raw **Canada immigration workbook** for the visualization-focused stages of the project.  
The goal is to create a **transparent, validated, and reusable analytical dataset** for the later exploratory, statistical, and geospatial notebooks.

The source workbook reports annual immigration flows to Canada by citizenship/origin from **1980 through 2013** and includes geographic classifications such as continent, region, and development group.

### Data-preparation questions

1. What is contained in the raw workbook and which worksheet should be used?
2. Which records belong in an international-origin analysis?
3. Which variables are analytically useful for visualization?
4. Are the annual immigration values complete, numeric, unique by origin, and non-negative?
5. What derived fields should be created for downstream analysis?
6. Can the cleaned dataset be exported as a single reproducible input for the rest of the project?

> **Source:** United Nations, *International Migration Flows to and from Selected Countries: The 2015 Revision*.  
> The workbook used here is the downloaded project file `Canada.xlsx`.


## Data Source and Metadata

The raw source file for this project is **`Canada.xlsx`**, which is derived from the United Nations international migration database.

### Source Metadata

| Metadata Field                         | Description                                                                       |
| -------------------------------------- | --------------------------------------------------------------------------------- |
| **Publisher**                          | United Nations, Department of Economic and Social Affairs, Population Division    |
| **Dataset Title**                      | *International Migration Flows to and from Selected Countries: The 2015 Revision* |
| **Database Identifier**                | `POP/DB/MIG/Flow/Rev.2015`                                                        |
| **Release**                            | December 2015                                                                     |
| **Reporting Country**                  | Canada                                                                            |
| **Classification Criterion**           | Citizenship                                                                       |
| **Observed Phenomenon**                | Annual immigration flows to Canada                                                |
| **Time Coverage Used in This Project** | 1980–2013                                                                         |
| **Frequency**                          | Annual                                                                            |
| **Primary Unit**                       | Number of immigrants                                                              |
| **Geographic Dimensions**              | Country/origin, major area (continent), region, development region                |
| **Project Raw File**                   | `Canada.xlsx`                                                                     |
| **Worksheet Used**                     | `Canada by Citizenship`                                                           |

The workbook provides the following suggested citation:

> United Nations, Department of Economic and Social Affairs, Population Division (2015). *International Migration Flows to and from Selected Countries: The 2015 Revision*. United Nations database, `POP/DB/MIG/Flow/Rev.2015`.

### What One Row Represents

In the worksheet used for analysis, each substantive row represents an **origin/citizenship classification associated with immigration to Canada**, together with:

* a broad geographic area;
* a subregion;
* a developed/developing-region classification; and
* annual immigrant counts for each year from 1980 through 2013.

The raw worksheet contains **197 rows and 43 columns** after the 20-line metadata header is skipped. These rows are not all independent foreign-origin countries, however. The workbook also contains special records such as:

* **`Canada`**, classified under `Citizens`;
* **`Unknown`**, for cases without an identified geographic origin; and
* **`Total`**, an aggregate record covering the entire table.

These observations are handled explicitly later in this notebook so that the final analytical dataset has a clearly defined scope.

### Important Interpretation Note

The field named `Country` in the cleaned dataset is best interpreted as the **reported citizenship/origin category in the UN source**, rather than as a measure of place of birth, ethnicity, or current nationality in a broader demographic sense.

Accordingly, the annual values used in this project describe recorded **immigration flows to Canada by citizenship/origin classification**. They should not be interpreted as Canada's total resident immigrant population or immigrant stock.

### Why This Dataset Is Suitable for the Project

The source combines several complementary analytical dimensions:

**Time × Origin Country × Continent × Region × Development Classification × Immigration Volume**

This structure makes the dataset particularly suitable for a Python visualization portfolio because it supports:

* longitudinal trend analysis;
* country and regional comparisons;
* distributional analysis;
* compositional visualization;
* statistical graphics; and
* geospatial mapping.

The goal of Notebook 01 is therefore to preserve these analytically useful dimensions while removing administrative fields and observations that do not belong in the international-origin analysis.


## 1. Setup and source file

The notebook reads directly from the downloaded workbook from United Nations website.  
For portability, it checks both the repository root and a conventional `data/` directory.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.0f}" if pd.notna(x) else "")


In [2]:
DATA_CANDIDATES = [
    Path("Canada.xlsx"),
    Path("data") / "Canada.xlsx",
]

DATA_PATH = next((path for path in DATA_CANDIDATES if path.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        "Canada.xlsx was not found. Place it in the notebook directory "
        "or in a data/ subdirectory."
    )

print(f"Using source workbook: {DATA_PATH.resolve()}")


Using source workbook: /mnt/data/Canada.xlsx


## 2. Inspect the workbook and load the raw migration table

The workbook contains multiple sheets. The project loads the records in sheet [Canada by Citizenship] and identifies aggregate/non-geographic observations explicitly so that every scope decision is visible.


In [3]:
excel_file = pd.ExcelFile(DATA_PATH)
print("Workbook sheets:")
for sheet in excel_file.sheet_names:
    print(f"  - {sheet}")


Workbook sheets:
  - Regions by Citizenship
  - Canada by Citizenship
  - Canada by Citizenship (2)


In [4]:
df_raw = pd.read_excel(
    DATA_PATH,
    sheet_name="Canada by Citizenship",
    skiprows=20,
)

print(f"Raw table shape: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns")
display(df_raw.head())


Raw table shape: 197 rows × 43 columns


,Type,Coverage,OdName,AREA,AreaName,REG,RegName,DEV,DevName,1980,1981,1982,1983,1984,1985,1986,1987,1988,1989,1990,1991,1992,1993,1994,1995,1996,1997,1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013
0,Immigrants,Foreigners,Afghanistan,935,Asia,5501,Southern Asia,902,Developing regions,16,39,39,47,71,340,496,741,828,1076,1028,1378,1170,713,858,1537,2212,2555,1999,2395,3326,4067,3697,3479,2978,3436,3009,2652,2111,1746,1758,2203,2635,2004
1,Immigrants,Foreigners,Albania,908,Europe,925,Southern Europe,901,Developed regions,1,0,0,0,0,0,1,2,2,3,3,21,56,96,71,63,113,307,574,1264,1816,1602,1021,853,1450,1223,856,702,560,716,561,539,620,603
2,Immigrants,Foreigners,Algeria,903,Africa,912,Northern Africa,902,Developing regions,80,67,71,69,63,44,69,132,242,434,491,872,795,717,595,1106,2054,1842,2292,2389,2867,3418,3406,3072,3616,3626,4807,3623,4005,5393,4752,4325,3774,4331
3,Immigrants,Foreigners,American Samoa,909,Oceania,957,Polynesia,902,Developing regions,0,1,0,0,0,0,0,1,0,1,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0
4,Immigrants,Foreigners,Andorra,908,Europe,925,Southern Europe,901,Developed regions,0,0,0,0,0,0,2,0,0,0,3,0,1,0,0,0,0,0,2,0,0,1,0,2,0,0,1,1,0,0,0,0,1,1


### Initial structure

The raw table combines:

- record descriptors (`Type`, `Coverage`);
- country/origin labels (`OdName`);
- numeric geographic codes (`AREA`, `REG`, `DEV`);
- readable geographic classifications (`AreaName`, `RegName`, `DevName`);
- 34 annual immigration-count columns (1980–2013).

Before cleaning, we inspect record types and the final rows because the workbook contains observations that should not automatically be treated as countries.


In [5]:
display(
    df_raw[["Type", "Coverage", "OdName", "AreaName", "RegName", "DevName"]]
    .tail(8)
)

print("\nCoverage values:")
display(df_raw["Coverage"].value_counts(dropna=False).rename("rows").to_frame())


,Type,Coverage,OdName,AreaName,RegName,DevName
189,Immigrants,Foreigners,Venezuela (Bolivarian Republic of),Latin America and the Caribbean,South America,Developing regions
190,Immigrants,Foreigners,Viet Nam,Asia,South-Eastern Asia,Developing regions
191,Immigrants,Foreigners,Western Sahara,Africa,Northern Africa,Developing regions
192,Immigrants,Foreigners,Yemen,Asia,Western Asia,Developing regions
193,Immigrants,Foreigners,Zambia,Africa,Eastern Africa,Developing regions
194,Immigrants,Foreigners,Zimbabwe,Africa,Eastern Africa,Developing regions
195,Immigrants,Foreigners,Unknown,World,World,World
196,Immigrants,Both,Total,World,World,World



Coverage values:


,rows
Coverage,
Foreigners,195
Citizens,1
Both,1


## 3. Define the analytical scope

This portfolio analyzes **identified international origins of immigrants to Canada**.

Three records require explicit treatment:

- **`Total`** is a workbook aggregate and would double-count the data if treated as an origin.
- **`Unknown`** has no identifiable geographic origin and therefore cannot support country-, region-, or map-based analysis.
- **`Canada`** is classified as `Citizens`, whereas the project focuses on international (`Foreigners`) origins.

These records are retained in the raw dataframe for auditability, but excluded from the analysis-ready dataset.


In [6]:
scope_audit = df_raw.loc[
    (df_raw["OdName"].isin(["Total", "Unknown", "Canada"]))
    | (df_raw["Coverage"] != "Foreigners"),
    ["Type", "Coverage", "OdName", "AreaName", "RegName", "DevName"],
].drop_duplicates()

display(scope_audit)


,Type,Coverage,OdName,AreaName,RegName,DevName
32,Immigrants,Citizens,Canada,Northern America,Northern America,Developed regions
195,Immigrants,Foreigners,Unknown,World,World,World
196,Immigrants,Both,Total,World,World,World


In [7]:
df = (
    df_raw.loc[
        (df_raw["Type"] == "Immigrants")
        & (df_raw["Coverage"] == "Foreigners")
        & (~df_raw["OdName"].isin(["Unknown", "Total"]))
    ]
    .copy()
)

print(f"Rows retained for international-origin analysis: {len(df)}")


Rows retained for international-origin analysis: 194


## 4. Select and standardize analytical variables

The geographic codes (`AREA`, `REG`, `DEV`) duplicate the readable geographic labels and are not needed for the planned visual analyses.  
The cleaned dataset therefore keeps interpretable labels and annual values, with clearer column names.

The final analytical dimensions are:

| Field | Meaning |
|---|---|
| `Country` | Identified country/territory of citizenship/origin |
| `Continent` | Broad geographic area |
| `Region` | Subregional classification |
| `Development_Group` | Developed / developing region classification |
| `1980`–`2013` | Annual immigrant counts |
| `Total` | Sum of annual counts across 1980–2013 |


In [8]:
YEARS = [str(year) for year in range(1980, 2014)]

# Standardize year-column labels first so downstream notebooks use one convention.
df.columns = [str(col) if isinstance(col, (int, np.integer)) else col for col in df.columns]

df = (
    df.rename(
        columns={
            "OdName": "Country",
            "AreaName": "Continent",
            "RegName": "Region",
            "DevName": "Development_Group",
        }
    )
    .drop(columns=["Type", "Coverage", "AREA", "REG", "DEV"])
)

# Reorder to make metadata dimensions appear before the annual measures.
df = df[
    ["Country", "Continent", "Region", "Development_Group"] + YEARS
]

display(df.head())


,Country,Continent,Region,Development_Group,1980,1981,1982,1983,1984,1985,1986,1987,1988,1989,1990,1991,1992,1993,1994,1995,1996,1997,1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013
0,Afghanistan,Asia,Southern Asia,Developing regions,16,39,39,47,71,340,496,741,828,1076,1028,1378,1170,713,858,1537,2212,2555,1999,2395,3326,4067,3697,3479,2978,3436,3009,2652,2111,1746,1758,2203,2635,2004
1,Albania,Europe,Southern Europe,Developed regions,1,0,0,0,0,0,1,2,2,3,3,21,56,96,71,63,113,307,574,1264,1816,1602,1021,853,1450,1223,856,702,560,716,561,539,620,603
2,Algeria,Africa,Northern Africa,Developing regions,80,67,71,69,63,44,69,132,242,434,491,872,795,717,595,1106,2054,1842,2292,2389,2867,3418,3406,3072,3616,3626,4807,3623,4005,5393,4752,4325,3774,4331
3,American Samoa,Oceania,Polynesia,Developing regions,0,1,0,0,0,0,0,1,0,1,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0
4,Andorra,Europe,Southern Europe,Developed regions,0,0,0,0,0,0,2,0,0,0,3,0,1,0,0,0,0,0,2,0,0,1,0,2,0,0,1,1,0,0,0,0,1,1


## 5. Validate data quality

Before deriving analytical measures, the dataset is checked for the conditions required by the later visualization notebooks:

- all expected years are present;
- country/origin labels are unique;
- key classification fields have no missing values;
- annual values are numeric;
- annual values are non-negative.

These checks are written as assertions so that a future change in the raw file fails visibly rather than producing silent errors.


In [9]:
expected_years = set(YEARS)
actual_years = set(YEARS).intersection(df.columns)

assert actual_years == expected_years, "One or more annual columns from 1980–2013 are missing."
assert not df["Country"].duplicated().any(), "Duplicate country/origin labels detected."
assert not df[["Country", "Continent", "Region", "Development_Group"]].isna().any().any(), (
    "Missing values detected in key classification fields."
)

# Convert annual values explicitly and validate them.
df[YEARS] = df[YEARS].apply(pd.to_numeric, errors="raise")

assert not df[YEARS].isna().any().any(), "Missing values detected in annual immigration counts."
assert (df[YEARS] >= 0).all().all(), "Negative immigration counts detected."

validation_summary = pd.DataFrame(
    {
        "Check": [
            "Identified international origins",
            "Annual columns",
            "Duplicate country labels",
            "Missing annual values",
            "Negative annual values",
        ],
        "Result": [
            len(df),
            len(YEARS),
            int(df["Country"].duplicated().sum()),
            int(df[YEARS].isna().sum().sum()),
            int((df[YEARS] < 0).sum().sum()),
        ],
    }
)

display(validation_summary)


,Check,Result
0,Identified international origins,194
1,Annual columns,34
2,Duplicate country labels,0
3,Missing annual values,0
4,Negative annual values,0


## 6. Create analysis-ready measures

A `Total` field summarizes cumulative immigration from each origin over the full 1980–2013 period.  
This is useful for ranking, composition, distribution, and geospatial analyses in later notebooks.

The annual columns remain intact so that no time-series information is lost.


In [10]:
df["Total"] = df[YEARS].sum(axis=1)

# Set Country as the analytical index while retaining the column on export.
df_can = df.set_index("Country").sort_index()

print(f"Analysis-ready shape: {df_can.shape[0]} origins × {df_can.shape[1]} variables")
display(df_can.head())


Analysis-ready shape: 194 origins × 38 variables


,Continent,Region,Development_Group,1980,1981,1982,1983,1984,1985,1986,1987,1988,1989,1990,1991,1992,1993,1994,1995,1996,1997,1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,Total
Country,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Afghanistan,Asia,Southern Asia,Developing regions,16,39,39,47,71,340,496,741,828,1076,1028,1378,1170,713,858,1537,2212,2555,1999,2395,3326,4067,3697,3479,2978,3436,3009,2652,2111,1746,1758,2203,2635,2004,58639
Albania,Europe,Southern Europe,Developed regions,1,0,0,0,0,0,1,2,2,3,3,21,56,96,71,63,113,307,574,1264,1816,1602,1021,853,1450,1223,856,702,560,716,561,539,620,603,15699
Algeria,Africa,Northern Africa,Developing regions,80,67,71,69,63,44,69,132,242,434,491,872,795,717,595,1106,2054,1842,2292,2389,2867,3418,3406,3072,3616,3626,4807,3623,4005,5393,4752,4325,3774,4331,69439
American Samoa,Oceania,Polynesia,Developing regions,0,1,0,0,0,0,0,1,0,1,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,6
Andorra,Europe,Southern Europe,Developed regions,0,0,0,0,0,0,2,0,0,0,3,0,1,0,0,0,0,0,2,0,0,1,0,2,0,0,1,1,0,0,0,0,1,1,15


## 7. Sanity checks for downstream visual analysis

The following summaries are not intended as full exploratory analysis—that begins in Notebook 02.  
They simply verify that the prepared data behave as expected and provide useful reference points for subsequent visualizations.


In [11]:
summary = pd.DataFrame(
    {
        "Metric": [
            "Origins",
            "Continents",
            "Regions",
            "Years",
            "First year",
            "Last year",
            "Total identified international immigrants, 1980–2013",
        ],
        "Value": [
            df_can.shape[0],
            df_can["Continent"].nunique(),
            df_can["Region"].nunique(),
            len(YEARS),
            YEARS[0],
            YEARS[-1],
            int(df_can["Total"].sum()),
        ],
    }
)

display(summary)


,Metric,Value
0,Origins,194
1,Continents,6
2,Regions,22
3,Years,34
4,First year,1980
5,Last year,2013
6,"Total identified international immigrants, 198...",6409133


In [12]:
top_origins = (
    df_can[["Continent", "Region", "Development_Group", "Total"]]
    .sort_values("Total", ascending=False)
    .head(10)
)

display(top_origins)


,Continent,Region,Development_Group,Total
Country,,,,
India,Asia,Southern Asia,Developing regions,691904
China,Asia,Eastern Asia,Developing regions,659962
United Kingdom of Great Britain and Northern Ireland,Europe,Northern Europe,Developed regions,551500
Philippines,Asia,South-Eastern Asia,Developing regions,511391
Pakistan,Asia,Southern Asia,Developing regions,241600
United States of America,Northern America,Northern America,Developed regions,241122
Iran (Islamic Republic of),Asia,Southern Asia,Developing regions,175923
Sri Lanka,Asia,Southern Asia,Developing regions,148358
Republic of Korea,Asia,Eastern Asia,Developing regions,142581


## 8. Export the cleaned dataset

All subsequent notebooks should use the exported dataset rather than repeating the preprocessing logic.

The CSV preserves:

- one row per identified international origin;
- geographic classification fields;
- annual immigration counts for 1980–2013;
- the derived `Total` measure.

This establishes a reproducible project pipeline:

**`Canada.xlsx` → Notebook 01 → cleaned CSV → visualization notebooks**


In [13]:
OUTPUT_DIR = Path("data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = OUTPUT_DIR / "canada_immigration_clean.csv"

df_can.reset_index().to_csv(OUTPUT_PATH, index=False)

print(f"Cleaned dataset saved to: {OUTPUT_PATH}")
print(f"Exported shape: {df_can.shape[0]} rows × {df_can.reset_index().shape[1]} columns")


Cleaned dataset saved to: data/canada_immigration_clean.csv
Exported shape: 194 rows × 39 columns


In [14]:
# Re-read the exported file as a final reproducibility check.
df_check = pd.read_csv(OUTPUT_PATH)

assert df_check.shape == df_can.reset_index().shape
assert df_check["Country"].is_unique
assert df_check[YEARS].isna().sum().sum() == 0

print("Export verification passed.")
display(df_check.head(3))


Export verification passed.


,Country,Continent,Region,Development_Group,1980,1981,1982,1983,1984,1985,1986,1987,1988,1989,1990,1991,1992,1993,1994,1995,1996,1997,1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,Total
0,Afghanistan,Asia,Southern Asia,Developing regions,16,39,39,47,71,340,496,741,828,1076,1028,1378,1170,713,858,1537,2212,2555,1999,2395,3326,4067,3697,3479,2978,3436,3009,2652,2111,1746,1758,2203,2635,2004,58639
1,Albania,Europe,Southern Europe,Developed regions,1,0,0,0,0,0,1,2,2,3,3,21,56,96,71,63,113,307,574,1264,1816,1602,1021,853,1450,1223,856,702,560,716,561,539,620,603,15699
2,Algeria,Africa,Northern Africa,Developing regions,80,67,71,69,63,44,69,132,242,434,491,872,795,717,595,1106,2054,1842,2292,2389,2867,3418,3406,3072,3616,3626,4807,3623,4005,5393,4752,4325,3774,4331,69439


## Preparation outcome

Notebook 01 produces a **validated, analysis-ready country-level dataset** for the remainder of the project.

### Key preparation decisions

- Used the original `Canada.xlsx` workbook as the authoritative raw source.
- Restricted the analytical dataset to identified international (`Foreigners`) origins.
- Standardized annual column labels to strings for consistent downstream selection.
- Added a cumulative `Total` measure without discarding annual detail.
- Added assertions for uniqueness, completeness, numeric validity, and non-negative counts.
- Exported one clean CSV to serve as the common input for all later visualization notebooks.

**Next:** Notebook 02 will use this prepared dataset for question-driven exploratory visual analytics.
